# SEEP: task-specific CLS steering — CUB200, seed 3407

Bật **một GPU** và **Internet**, dùng **Add Input → trungkiennguyn/lgsp-dataset**
trước **Save Version**, rồi Run All. Notebook có thể tự gắn dataset qua
kagglehub trong session tương tác. Không cần upload source hay push branch.

Một run mới: 10 task × 30 epoch SEEP + 5 epoch align-only mỗi task.
CLS Q/K/V tại block 9/10/11; không transport hay module/loss khác.
Raw_seen (C) và align-only (D) dùng cùng feature; A/B lấy từ ZIP đã audit.
Mốc baseline lịch sử giữ riêng. Ước lượng cũ T4 khoảng 3 giờ chỉ để dự trù;
notebook đo thời gian thực. Các tests CPU không phải kết quả full CUB.

Ý tưởng additive CLS Q/K/V: [CASP §4.1](https://arxiv.org/html/2601.16773v1),
Huang, Lin, Lu (2026); adaptation task-specific cho SEEP CIL, chưa xác lập novelty.
Routing gốc chọn một task cho mỗi batch. Không dùng test để chọn LR/layer.


In [ ]:
import os, sys, json, subprocess, hashlib, base64, shutil, zipfile, time
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Use one device even on Kaggle T4 x2.
import torch, torchvision

REPO = Path('/kaggle/working/SEEP_CLS_STEERING')
READY = Path('/kaggle/working/cub200_ready')
OUTPUT = Path('/kaggle/working/seep_cls_steering_align_only_seed3407')
PREFLIGHT = Path('/kaggle/working/cls_steering_preflight')
DATASET_HANDLE = 'trungkiennguyn/lgsp-dataset'
DATA_ROOT = None  # Chỉ cần điền nếu Input chứa nhiều official CUB roots.
RESUME = False
RESUME_FROM = None  # Optional: thư mục output cũ, gồm checkpoint/dgca_latest.pth.
assert torch.cuda.is_available(), 'Bật GPU trong Settings của notebook'
assert torch.cuda.device_count() == 1, 'Chọn một GPU cho protocol này'
assert not OUTPUT.exists() or RESUME, 'Output đã tồn tại; chỉ dùng RESUME cho chính run này.'
if RESUME_FROM is not None:
    assert RESUME and not OUTPUT.exists()
    shutil.copytree(Path(RESUME_FROM), OUTPUT)
print('Python:', sys.version, 'Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Historical stack match:', torch.__version__ == '2.11.0+cu128' and torchvision.__version__ == '0.26.0+cu128')
print('Khác environment không được gọi là exact reproduction.')
PREFLIGHT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Giữ nguyên torch/torchvision Kaggle; không cài requirements.txt.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', 'timm==0.6.11'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'loguru', 'pytest', 'pyyaml', 'kagglehub'])
import importlib.metadata
maxxvit = Path(importlib.metadata.distribution('timm').locate_file('timm/models/maxxvit.py'))
original = maxxvit.read_text()
fixed = original.replace('from dataclasses import dataclass, replace\n',
                         'from dataclasses import dataclass, replace, field\n')
fixed = fixed.replace('conv_cfg: MaxxVitConvCfg = MaxxVitConvCfg()\n',
                      'conv_cfg: MaxxVitConvCfg = field(default_factory=MaxxVitConvCfg)\n')
fixed = fixed.replace('transformer_cfg: MaxxVitTransformerCfg = MaxxVitTransformerCfg()\n',
                      'transformer_cfg: MaxxVitTransformerCfg = field(default_factory=MaxxVitTransformerCfg)\n')
maxxvit.write_text(fixed)
TIMM_COMPAT = {'file': str(maxxvit), 'before_sha256': hashlib.sha256(original.encode()).hexdigest(),
               'after_sha256': hashlib.sha256(fixed.encode()).hexdigest(), 'idempotent': True}
subprocess.check_call([sys.executable, '-c', "import timm; assert timm.__version__ == '0.6.11'; print('timm OK')"])


In [ ]:
BASE_COMMIT = '80081081ab42769d102f83d447705da0639e2fb1'
PATCH_B64 = '...'
DELIVERY = {'base_commit': '80081081ab42769d102f83d447705da0639e2fb1', 'implementation_source_commit': '00147efdd8268e514f623432d7f30808868609b2', 'branch': 'feat/cls-steering-cil', 'patch_sha256': '5b040364f13374a19f896864d74cc6992174256bb97eb1073cb4d588ec3264d7', 'audited_dgca_patch_sha256': 'ecec522a42518819d2504890b6c7b6fede6938c5aca7d5d47322c2cd6cc9996e', 'source_files_sha256': {'.gitattributes': '8f51ea90a7a234f57e7c605dfbd317d84c85211e56a308f24d6ceadd73e9c6e6', '.gitignore': '96f7842956120b30446d23003c6cf57fe0e31b0d02dff13ee4b3233644a97733', 'README_CLS_STEERING.md': '65fa9a284acd7ea6707b8c9c40f50d8855035ba101a350338815ed682ee46126', 'README_DGCA.md': 'ed169cfca92f18ff1cbfd09b3d7a1a1f14d111f8043e472457cad25ede72a573', 'attention.py': 'dbdeff73e485903b0d2dd37b4f2ae964f1dc042afbc354b64bedbc30cbb9e098', 'configs/cub200_10task.py': 'f0ff7604e9c4ae2f1739fb1a214a184c5afc004db05d170802d2307936603c0e', 'continual_datasets/continual_datasets.py': '89d9ed00a4c273ef131bbdef79ec394fd903e20ed1ce62b1c8c36aecf9b7c8d1', 'datasets.py': 'd9a75e0685cf6f2b95a6af57f4be854c12e647e9e71bafed30a95e06eb43f835', 'engine.py': 'ad18248616085ee608b4af15828a70ea0b35c5574e03ad44869830f32af3fd5a', 'main.py': 'd1a009ae93c92f603d0c896c0c3fb009602beab502e1ac4ae6062aa83584d25c', 'modules/cls_diagnostics.py': 'd14eb29db3223d34c17851911196cb373b5d91870986586469af121dfa039a9f', 'modules/cls_pretrained.py': 'c140a81286e3e1daa44dfbdc24e08b373cdd38e32f9a8afc47f6ca3d4fa4bf67', 'modules/cls_steering.py': '28762a40ab39e2629a0e78ff60725cbddfb5f01edec2857b5b27d14ec386e203', 'modules/dgca.py': '0580058d687cba11745ce69f72560126aa90a7585bcd043b20d53c9f79100acd', 'modules/dgca_data.py': '3ef5c119c4e2193d4d96eaddd116927ec926bed8dfdba10031ac3739dc64095e', 'modules/dgca_runtime.py': 'd82833f0d9644000369a70a92ed0ae0ea463b86184ec6147b45a6b8df79a4c83', 'references/align_only_seed3407/dgca_implementation.patch': 'ecec522a42518819d2504890b6c7b6fede6938c5aca7d5d47322c2cd6cc9996e', 'references/align_only_seed3407/official_split_manifest.json': 'ccf1a0227ee8f3e72bebb73fce7896b79ea48b784c1a44770a8a5f78cfcd391f', 'references/align_only_seed3407/provenance.json': 'c5b0ed82374e3016b2cc56a6079bcd088ce943ed574034da8aac300e3ae40bff', 'references/align_only_seed3407/task_accuracy_matrix_dgca.csv': 'f341a173ca0c3a372ebe151900f4a0e309c06a5ab662afd7805069a6d60d003c', 'references/align_only_seed3407/task_accuracy_matrix_raw_seen.csv': '8ca1ce9a02f4ce7e0658a7583f7ecb2410623f39e185ac8aa8e69683736a3b2a', 'tests/test_cls_steering.py': '6865759bb03b02fbb126a4fdc22ba655355bbeb17aea30ac14abe593712b3cad', 'tests/test_dgca.py': 'ff0d03fc5062820bcf43175b2ec3352868b387408bcfe8f05683eb4bc8210415', 'tools/build_cls_delivery.py': 'bac5e6d42fc357980754114bbd44d48d4a2146fe9c384bc5b063248b581a4357', 'tools/build_dgca_delivery.py': '1406f1dc6b672fc6682cc4fbcdc7a15f6d63659544555eeab6e78dd7216cb394', 'tools/preflight_dgca.py': '0e56d15cae3196eeb672046ee42b2fb7b66c42dee5b5ac477369487cfc405cd1', 'tools/prepare_cub200.py': '5ec826cd800bc9a2038c9dd432c5b589d2f5696c8470a47acbdac5d53bad1ee2', 'tools/smoke_dgca.py': '5bdd6be5f3288ae7cd076ba16b8193e991165178acc62968d386ec7e4f425472', 'tools/summarize_cls_steering.py': 'e3e391850547eeb511b83decd79300242e681fd09b47a783723b475fb5755b30', 'tools/summarize_dgca.py': 'a82ca711a1ccfa7a001d0d05a6fdf5f0bef7d97381ad1fdd844f30573f81e4f2', 'tools/verify_cls_delivery.py': '6d89c8f0ec97e2908a84c04a2a5dce3cb25bfd42df8db9bef0589caa10b92b66', 'vision_transformer.py': 'de78992926f301995b4cb316ed2e6228950395b59cfaff7d6e860423eb4558e1'}, 'reference': {'archive_sha256': '570a0e4705202f369ab72d265c4d3022d00a73682d5829b1c710d7bdf2f11f80', 'files_sha256': {'task_accuracy_matrix_raw_seen.csv': '8ca1ce9a02f4ce7e0658a7583f7ecb2410623f39e185ac8aa8e69683736a3b2a', 'task_accuracy_matrix_dgca.csv': 'f341a173ca0c3a372ebe151900f4a0e309c06a5ab662afd7805069a6d60d003c', 'official_split_manifest.json': 'ccf1a0227ee8f3e72bebb73fce7896b79ea48b784c1a44770a8a5f78cfcd391f'}, 'split_sha256': '7f1cbc969854e73a21fba66461422d852d77405aa71dfbf9712d5f74de825e40', 'pretrained_sha256': 'ac7478131e1b77e5c2f0225849730bd19ee89cfffe600c2dfd2272b34a50a43d', 'dgca_patch_sha256': 'ecec522a42518819d2504890b6c7b6fede6938c5aca7d5d47322c2cd6cc9996e'}, 'delivery_mode': 'embedded_combined_patch_on_pinned_base', 'local_full_cub_run': False, 'local_gpu_available': False}
PATCH_SHA256 = DELIVERY['patch_sha256']
patch = base64.b64decode(PATCH_B64)
assert hashlib.sha256(patch).hexdigest() == PATCH_SHA256
if not REPO.exists():
    subprocess.check_call(['git', 'clone', '--config', 'core.autocrlf=false', 'https://github.com/thanhphu25/SEEP.git', str(REPO)])
    subprocess.check_call(['git', 'checkout', '--detach', BASE_COMMIT], cwd=REPO)
    subprocess.check_call(['git', 'switch', '-c', 'feat/cls-steering-cil'], cwd=REPO)
    patch_file = REPO/'cls_steering_implementation.patch'
    patch_file.write_bytes(patch)
    subprocess.check_call(['git', 'apply', '--check', str(patch_file)], cwd=REPO)
    subprocess.check_call(['git', 'apply', str(patch_file)], cwd=REPO)
else:
    assert RESUME, 'Repo đã tồn tại; dùng session mới hoặc RESUME cho chính run này'
    assert (REPO/'cls_steering_implementation.patch').read_bytes() == patch
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO).decode().strip() == BASE_COMMIT
for name, expected in DELIVERY['source_files_sha256'].items():
    assert hashlib.sha256((REPO/name).read_bytes()).hexdigest() == expected, name
(REPO/'cls_steering_delivery.json').write_text(json.dumps(DELIVERY, indent=2))
(REPO/'timm_compatibility.json').write_text(json.dumps(TIMM_COMPAT, indent=2))
print('Verified source:', DELIVERY['implementation_source_commit'], 'patch:', PATCH_SHA256)


In [ ]:
def find_cub(base):
    base = Path(base)
    return sorted({p.parent for p in base.rglob('images.txt')
                   if all((p.parent/n).is_file() for n in ('image_class_labels.txt', 'train_test_split.txt'))
                   and (p.parent/'images').is_dir()}) if base.exists() else []
if DATA_ROOT is None:
    candidates = find_cub('/kaggle/input')
    if not candidates:
        import kagglehub
        try:
            attached = kagglehub.dataset_download(DATASET_HANDLE)
        except Exception as exc:
            raise RuntimeError('Dùng Add Input để gắn '+DATASET_HANDLE+' trước Save Version, rồi chạy lại.') from exc
        candidates = find_cub(attached) or find_cub('/kaggle/input')
    print('CUB candidates:', [str(p) for p in candidates])
    assert len(candidates) == 1, 'Cần đúng một official CUB root; đặt DATA_ROOT trong cell cấu hình.'
    DATA_ROOT = candidates[0]
subprocess.check_call([sys.executable, 'tools/prepare_cub200.py', '--source', str(DATA_ROOT),
                       '--output', str(READY)], cwd=REPO)
official = json.loads((READY/'official_split_manifest.json').read_text())
reference_official = json.loads((REPO/'references/align_only_seed3407/official_split_manifest.json').read_text())
assert official['official_metadata_sha256'] == reference_official['official_metadata_sha256']
assert (official['image_count'], official['train'], official['test'], official['classes']) == (11788, 5994, 5794, 200)
print(json.dumps(official, indent=2))


In [ ]:
ARGS = ['--seed', '3407', '--batch-size', '32', '--epochs', '30', '--num_tasks', '10',
        '--num_tasks_to_run', '0', '--lgsp', 'YES', '--lgsp_type', 'LGSP',
        '--data-path', str(READY)+'/', '--output_dir', str(OUTPUT), '--dgca', 'align_only',
        '--dgca-shrinkage', '0.1', '--dgca-samples-per-class', '128', '--dgca-epochs', '5', '--dgca-lr', '0.01',
        '--cls-steering', 'task', '--cls-steering-layers', '9', '10', '11', '--cls-steering-lr', '0.0005']
RUN_ENV = os.environ.copy()
RUN_ENV['SEEP_PRETRAINED_SHA256'] = DELIVERY['reference']['pretrained_sha256']
def run_logged(command, log_path):
    start = time.perf_counter()
    with Path(log_path).open('w') as log:
        process = subprocess.Popen(command, cwd=REPO, env=RUN_ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return {'returncode': returncode, 'seconds': time.perf_counter()-start, 'command': command}
TEST_LOG = PREFLIGHT/'tests.log'
TEST_RESULT = run_logged([sys.executable, '-m', 'pytest', 'tests/test_dgca.py', 'tests/test_cls_steering.py',
                          '-q', '--junitxml='+str(PREFLIGHT/'tests.xml')], TEST_LOG)
(PREFLIGHT/'test_result.json').write_text(json.dumps(TEST_RESULT, indent=2))


In [ ]:
PREFLIGHT_RESULT = run_logged([sys.executable, 'tools/preflight_dgca.py', *ARGS,
                               '--report-dir', str(PREFLIGHT)], PREFLIGHT/'preflight.log')
resolved = json.loads((PREFLIGHT/'resolved_config.json').read_text())
assert resolved['dgca'] == 'align_only' and resolved['cls_steering'] == 'task'
assert resolved['effective_lr'] == .00375 and resolved['cls_steering_lr'] == .0005
assert resolved['cls_steering_layers'] == [9, 10, 11]
assert resolved['num_tasks_to_run'] == 0 and resolved['task_inc'] is False
assert json.loads((PREFLIGHT/'split_manifest.json').read_text())['sha256'] == DELIVERY['reference']['split_sha256']
weights = json.loads((PREFLIGHT/'pretrained_manifest.json').read_text())
assert weights['sha256'] == DELIVERY['reference']['pretrained_sha256'] and weights['tensor_load_verified']
# Separate process: exactly two current-task batches, no HP/accuracy selection.
SMOKE_RESULT = run_logged([sys.executable, 'tools/smoke_dgca.py', *ARGS], PREFLIGHT/'smoke.log')
(PREFLIGHT/'checks.json').write_text(json.dumps({'tests': TEST_RESULT, 'preflight': PREFLIGHT_RESULT,
                                              'gpu_smoke': SMOKE_RESULT}, indent=2))


Full run duy nhất. Khi tiếp tục checkpoint, dùng `RESUME=True` trong cấu hình;
ở session mới, `RESUME_FROM` trỏ đến thư mục output của chính run này đã gắn vào Input.
Checkpoint transport/align-only cũ bị từ chối. Không resume giữa epoch.


In [ ]:
if RESUME:
    assert (OUTPUT/'checkpoint/dgca_latest.pth').is_file(), 'Chưa có task-boundary checkpoint của run này'
else:
    assert not OUTPUT.exists(), 'Không ghi chồng run'
STREAM_LOG = PREFLIGHT/'full_stream.log'
RUN_RESULT = run_logged([sys.executable, '-u', 'main.py', 'cub200_10task', *ARGS,
                         *(['--dgca-resume'] if RESUME else [])], STREAM_LOG)
(PREFLIGHT/'full_run_result.json').write_text(json.dumps(RUN_RESULT, indent=2))


In [ ]:
# Chỉ gói metrics sau một full run thành công. Không yêu cầu raw C bằng A.
run_logged([sys.executable, 'tools/summarize_cls_steering.py', str(OUTPUT)], PREFLIGHT/'summary.log')
summary = json.loads((OUTPUT/'summary.json').read_text())
assert summary['full_protocol_complete'] and summary['completed_tasks'] == 10
assert json.loads((OUTPUT/'run_status.json').read_text())['status'] == 'success'
source = json.loads((OUTPUT/'source_manifest.json').read_text())
assert source['diff_sha256'] == PATCH_SHA256
assert source['cached_weights_sha256']['ViT-B_16.npz'] == DELIVERY['reference']['pretrained_sha256']
assert json.loads((OUTPUT/'split_manifest.json').read_text())['sha256'] == DELIVERY['reference']['split_sha256']
for name, expected in DELIVERY['source_files_sha256'].items():
    assert hashlib.sha256((REPO/name).read_bytes()).hexdigest() == expected, name
reports = [json.loads(p.read_text()) for p in sorted(OUTPUT.glob('dgca_task_*.json'))]
assert len(reports) == 10
assert all(r['mode'] == 'align_only' and r['transport'] is None and r['hashes_before'] == r['hashes_after'] for r in reports)
print(json.dumps({'comparisons_pp': summary['comparisons_pp'], 'screening_pass': summary['screening_pass'],
                  'interpretation': summary['interpretation']}, indent=2))
BUNDLE = Path('/kaggle/working/SEEP_CLS_STEERING_CUB200_metrics.zip')
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUTPUT.iterdir()):
        if p.is_file() and p.suffix in {'.json', '.csv', '.txt', '.log', '.patch'}:
            archive.write(p, 'metrics/'+p.name)
    for p in sorted(PREFLIGHT.iterdir()):
        if p.is_file():
            archive.write(p, 'preflight/'+p.name)
    for name in ('cls_steering_delivery.json', 'cls_steering_implementation.patch', 'timm_compatibility.json'):
        archive.write(REPO/name, 'source/'+name)
    for p in sorted((REPO/'references/align_only_seed3407').iterdir()):
        archive.write(p, 'reference/'+p.name)
    archive.write(READY/'official_split_manifest.json', 'data/official_split_manifest.json')
    notebook = Path('/kaggle/working/__notebook__.ipynb')
    if notebook.is_file():
        archive.write(notebook, 'notebook/executed.ipynb')
from IPython.display import display, FileLink
display(FileLink(str(BUNDLE)))
print('Checkpoint lớn, tải riêng:')
display(FileLink(str(OUTPUT/'checkpoint/dgca_latest.pth')))
